## Training and Evaluation

In [5]:
%pip install nbformat

Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   ---------------------------------------- 2/2 [nbformat]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
%%capture
%run Preprocessing.ipynb

In [2]:
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import  RobustScaler
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import  accuracy_score,f1_score, balanced_accuracy_score, classification_report

In [3]:
PCA_pipe = Pipeline ([("prep", ColumnTransformer([("num",RobustScaler(),num_cols)], remainder="passthrough")),("model",PCA(n_components=0.9))]) 
x_pca = PCA_pipe.fit_transform(x_tr)
x_pca_v = PCA_pipe.transform(x_v)

# Now our data is scaled and with reduced dimensions
def majority(y_tr,x_v):
   majority = np.unique_counts(y_tr).values[np.argmax(np.unique_counts(y_tr).counts)]
   pred = np.ones(len(x_v))
   return majority*pred
print(f"Majority class macro f1 score is {f1_score(y_v, majority(y_tr,x_pca_v),average="macro"):.2f}")

LR_model = LogisticRegression()
LR_model.fit(x_pca,y_tr)
print(f"Multinomial Logistic Regression macro f1 score is  {f1_score(y_v, LR_model.predict(x_pca_v),average="macro"):.2f} ")
Tree_model = DecisionTreeClassifier()
Tree_model.fit(x_pca,y_tr)
print(f"Decision Tree macro f1 score is  {f1_score(y_v, Tree_model.predict(x_pca_v),average="macro"):.2f} ")
RFC_model = RandomForestClassifier()
RFC_model.fit(x_pca,y_tr)
print(f"Random Forest macro f1 score is  {f1_score(y_v, RFC_model.predict(x_pca_v),average="macro"):.2f} ")

Majority class macro f1 score is 0.22
Multinomial Logistic Regression macro f1 score is  0.82 
Decision Tree macro f1 score is  0.81 
Random Forest macro f1 score is  0.67 


In [4]:
def cross_valid(model,folds):
    F1_scores = []
    for i in range(folds):
        x_tr,x_v,y_tr,y_v = train_test_split(x_trv,y_trv,random_state=10+i,stratify=y_trv,test_size=0.2)

        #For each fold we get a different x_tr and x_v split from the earlier x_trv(x_trv does not contain the test data)
        medians = [x_tr[col].median() for col in num_cols] #num_col and cat_col remain the same regardless of x_tr
        impute_num(x_tr,medians)
        impute_num(x_v,medians)
        dummy_cols = pd.get_dummies(x_tr,columns = cat_cols).columns
        x_tr = impute_col(x_tr, dummy_cols)
        x_v = impute_col(x_v, dummy_cols)

        PCA_pipe = Pipeline ([("prep", ColumnTransformer([("num",RobustScaler(),num_cols)], remainder="passthrough")),("model",PCA(n_components=0.9))])
        model.fit(PCA_pipe.fit_transform(x_tr),y_tr)
        pred = model.predict(PCA_pipe.transform(x_v))
        score = f1_score(y_v,pred,average="macro")
        F1_scores.append(score)

    return [np.mean(F1_scores), np.std(F1_scores)]

for i in [1,10,100]:
    print(cross_valid(RandomForestClassifier(n_estimators=i),4)[0])


0.5575476826237475
0.7190753486709369
0.7184741801408469


In [5]:
def evaluate(model, x_te, y_te):

    y_pred = model.predict(PCA_pipe.transform(x_te))

    acc = accuracy_score(y_te, y_pred)
    bal_acc = balanced_accuracy_score(y_te, y_pred)
    F1 = f1_score(y_te, y_pred, average="macro")
    per_class_prec = classification_report(y_te, y_pred)

    return acc, bal_acc, F1, per_class_prec

evaluate(LR_model, x_te, y_te)
evaluate(Tree_model, x_te, y_te)
evaluate(RFC_model, x_te, y_te)

(0.9230769230769231,
 0.8935672514619885,
 0.9150997150997151,
 '              precision    recall  f1-score   support\n\n         0.0       1.00      0.80      0.89         5\n         1.0       0.90      0.95      0.92        19\n         2.0       0.93      0.93      0.93        15\n\n    accuracy                           0.92        39\n   macro avg       0.94      0.89      0.92        39\nweighted avg       0.93      0.92      0.92        39\n')

In [6]:
x_num = x_tr[num_cols]
cat_cols_new = x_tr.columns[~np.isin(x_tr.columns,num_cols)] #different from original cat_cols due to OHE
x_cat = x_tr[cat_cols_new]
x_num_pca = PCA_pipe.fit_transform(x_num)
LR_model.fit(x_num_pca,y_tr)
x_num_v = PCA_pipe.transform(x_v[num_cols])
print(f"Numerical only LR F1 score is {f1_score(y_v,LR_model.predict(x_num_v),average="macro")}")

pca_cat = PCA(n_components=0.9) #We dont need to scale features so we dont use the PCA pipe
x_cat_pca = pca_cat.fit_transform(x_cat)
LR_model.fit(x_cat_pca,y_tr)
x_cat_v = pca_cat.transform(x_v[cat_cols_new])
print(f"Categorical only LR F1 score is {f1_score(y_v,LR_model.predict(x_cat_v),average="macro")}")

x_combined_pca = PCA_pipe.fit_transform(x_tr)
LR_model.fit(x_combined_pca,y_tr)
x_combined_v = PCA_pipe.transform(x_v)
print(f"Combined LR F1 score is {f1_score(y_v,LR_model.predict(x_combined_v),average="macro")}")

Numerical only LR F1 score is 0.8499525166191834
Categorical only LR F1 score is 0.6417432470064048
Combined LR F1 score is 0.8243114909781576
